#           EMAIL SPAM CLASSIFIER

#IMPORT

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
import os
import kagglehub

nltk.download('stopwords')

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


#LOAD DATA


In [ ]:
#LOAD DATA

In [2]:
#Load Data
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
print("Dataset Path:", path)

print("Files:", os.listdir(path))

file_path = os.path.join(path, "spam.csv")
df = pd.read_csv(file_path, encoding='latin-1')

# Clean columns

df = df[['v1', 'v2']]
df.columns = ['label', 'text']

# Remove duplicates

df.drop_duplicates(inplace=True)

print("\n📊 Dataset Info:")
print(df['label'].value_counts())

100%|██████████| 211k/211k [00:00<00:00, 28.7MB/s]

Extracting files...
Dataset Path: /root/.cache/kagglehub/datasets/uciml/sms-spam-collection-dataset/versions/1
Files: ['spam.csv']

📊 Dataset Info:
label
ham     4516
spam     653
Name: count, dtype: int64


#TEXT CLEANING (NLP)

In [3]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['cleaned'] = df['text'].apply(clean_text)

# Demo cleaning

sample = "Win ₹10000 NOW!!! Click here!!!"
print("\n🔍 Cleaning Demo:")
print("Original:", sample)
print("Cleaned :", clean_text(sample))


🔍 Cleaning Demo:
Original: Win ₹10000 NOW!!! Click here!!!
Cleaned : win click


#FEATURE ENGINEERING

In [4]:
vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1,2))
X = vectorizer.fit_transform(df['cleaned'])

encoder = LabelEncoder()
y = encoder.fit_transform(df['label'])  # spam=1

#SPLIT


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

#TRAIN MODELS

In [6]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(probability=True)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results[name] = {
        "model": model,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred)
    }



#  SHOW RESULTS



In [12]:
print("\nMODEL COMPARISON:\n")

for name, res in results.items():
    print(f"{name}:")
    print(f" Accuracy : {res['accuracy']:.4f}")
    print(f" Precision: {res['precision']:.4f}")
    print(f" Recall   : {res['recall']:.4f}")
    print(f" F1 Score : {res['f1']:.4f}")
    print("-" * 30)

# Best model (based on precision)

best_model_name = max(results, key=lambda x: results[x]['precision'])
best_model = results[best_model_name]['model']

print(f"\nBest Model: {best_model_name}")

print("\nInsight: Precision is prioritized to avoid misclassifying important emails as spam.")


MODEL COMPARISON:

Naive Bayes:
 Accuracy : 0.9816
 Precision: 1.0000
 Recall   : 0.8571
 F1 Score : 0.9231
------------------------------
Logistic Regression:
 Accuracy : 0.9681
 Precision: 0.9808
 Recall   : 0.7669
 F1 Score : 0.8608
------------------------------
SVM:
 Accuracy : 0.9797
 Precision: 0.9912
 Recall   : 0.8496
 F1 Score : 0.9150
------------------------------

Best Model: Naive Bayes

Insight: Precision is prioritized to avoid misclassifying important emails as spam.


#SPAM WORDS


In [9]:
feature_names = vectorizer.get_feature_names_out()
spam_indices = np.where(y == 1)[0]

spam_matrix = X[spam_indices].toarray()
avg_scores = np.mean(spam_matrix, axis=0)

top_indices = avg_scores.argsort()[-10:][::-1]

print("\n🔥 Top Global Spam Words:")
for i in top_indices:
    print(feature_names[i])


🔥 Top Global Spam Words:
call
free
txt
text
stop
mobile
claim
reply
prize
ur


#PREDICTION SYSTEM

In [13]:
def get_keywords(text):
    cleaned = clean_text(text)
    vector = vectorizer.transform([cleaned])

    scores = vector.toarray()[0]
    feature_names = vectorizer.get_feature_names_out()

    top_indices = scores.argsort()[-5:][::-1]
    keywords = [(feature_names[i], round(scores[i], 3)) for i in top_indices if scores[i] > 0]

    return keywords

def get_explanation(prob):
    if prob > 0.9:
        return "Strong spam indicators with high-risk promotional language."
    elif prob > 0.7:
        return "Moderate spam signals detected."
    else:
        return "Low spam probability, likely safe message."

def get_risk(prob):
    if prob > 0.9:
        return "HIGH 🚨"
    elif prob > 0.7:
        return "MEDIUM ⚠️"
    else:
        return "LOW ✅"

def predict_email(text):
    if text.strip() == "":
        print("⚠️ Please enter a valid message")
        return

    cleaned = clean_text(text)
    vector = vectorizer.transform([cleaned])

    pred = best_model.predict(vector)[0]
    prob = best_model.predict_proba(vector)[0][1]

    keywords = get_keywords(text)
    explanation = get_explanation(prob)
    risk = get_risk(prob)

    print("\n==============================")
    print(" Message:", text)
    print(" Prediction:", "SPAM 🚨" if pred == 1 else "HAM ✅")
    print(f" Probability: {prob*100:.2f}%")
    print(f" Risk Level: {risk}")

    print("\n Top Keywords:")
    for word, score in keywords:
        print(f" - {word} ({score})")

    print("\n Explanation:")
    print(explanation)
    print("==============================")

#TEST

In [11]:
predict_email("Congratulations! You won a FREE iPhone. Click now!")
predict_email("Hey, are we meeting at 5 PM today?")


📩 Message: Congratulations! You won a FREE iPhone. Click now!
🔍 Prediction: SPAM 🚨
📊 Probability: 54.71%
⚠️ Risk Level: LOW ✅

🧠 Top Keywords:
 - click (0.684)
 - congratulations (0.622)
 - free (0.38)

💡 Explanation:
Low spam probability, likely safe message.

📩 Message: Hey, are we meeting at 5 PM today?
🔍 Prediction: HAM ✅
📊 Probability: 1.38%
⚠️ Risk Level: LOW ✅

🧠 Top Keywords:
 - meeting (0.558)
 - pm (0.54)
 - hey (0.458)
 - today (0.433)

💡 Explanation:
Low spam probability, likely safe message.
